# 04 — Train: Ridge regression

Searches Ridge regularization for the configured target station's direct 24-hour water-level forecast over the joined feature artifacts, then evaluates the selected model once on the sealed test cohort.

**Inputs:** joined train/test feature artifacts and their metadata contract  
**Outputs:** in-notebook prediction preview/test metrics and an MLflow run hierarchy

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Imports the dependencies and loads the joined feature metadata. The target station's prefixed predictor columns are the source of truth for the model inputs, including its raw water level at issue time `t`. The Ridge alpha search and validation policy are explicit constants so every fold and MLflow run remains inspectable.

**Parameters**

| Parameter | Value | What it does |
| --- | --- | --- |
| `PROCESSED_DIR` | `data/processed/joined` | Directory the joined Stage-3 Parquets and metadata are read from. This notebook only reads — it never writes back. |
| `PREDICTION_PREVIEW_ROWS` | `5` | Number of scored test rows shown in the final preview. |
| `FEATURE_COLUMNS` | all target-station predictors | Every prefixed target-station predictor, including `{TARGET_STATION_ID}__water_level` at issue time `t`; metadata fields are excluded. |
| `TARGET_COLUMNS` | `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_24` | The 24 future water levels predicted directly from one issue-time feature vector. |
| `FORECAST_HORIZON_HOURS` | `24` | Number of direct future target outputs and the metadata contract width. |
| `RIDGE_ALPHAS` | `[0.01, 0.1, 1.0, 10.0, 100.0]` | Candidate L2 regularization strengths. |
| `N_VALIDATION_FOLDS` | `5` | Number of expanding-window validation folds. |
| `INITIAL_TRAIN_FRACTION` | `0.50` | Approximate fraction of eligible rows in the first fold's training window. |
| `EMBARGO_HOURS` | `24` | Number of rows left between each fold's training and validation windows. |
| `CV_SELECTION_METRIC` | `"mae"` | Aggregate CV metric used to select alpha; `"rmse"` is also supported. |
| `MLFLOW_EXPERIMENT_NAME` | `"ridge"` | Experiment receiving the parent, nested fold, and final test runs. |

In [ ]:
import json
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

from src.config import (
    CV_SELECTION_METRIC,
    EMBARGO_HOURS,
    INITIAL_TRAIN_FRACTION,
    MLFLOW_TRACKING_URI,
    N_VALIDATION_FOLDS,
    FORECAST_HORIZON_HOURS,
    TARGET_STATION_ID,
)
from src.metrics import metric_tables

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
PROCESSED_DIR = Path("data/processed/joined")
METADATA_PATH = PROCESSED_DIR / "all_stations_feature_metadata.json"
RIDGE_ALPHAS = [0.01, 0.1, 1.0, 10.0, 100.0]
MLFLOW_EXPERIMENT_NAME = "ridge"
PREDICTION_PREVIEW_ROWS = 5
if CV_SELECTION_METRIC not in {"mae", "rmse"}:
    raise ValueError("CV_SELECTION_METRIC must be either 'mae' or 'rmse'")
EXCLUDED_METADATA_FIELDS = {"timestamp", "station_id", "target_valid"}
feature_metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
station_id = TARGET_STATION_ID
if station_id not in feature_metadata["engineered_station_ids"]:
    raise ValueError(
        f"Target station {station_id!r} is not engineered in the joined feature metadata"
    )
TARGET_VALID_COLUMN = f"{station_id}__target_valid"
TARGET_COLUMNS = [
    f"{station_id}__target_t_plus_{offset:02d}"
    for offset in range(1, FORECAST_HORIZON_HOURS + 1)
]
metadata_target_columns = feature_metadata.get("target_columns")
if feature_metadata.get("configuration", {}).get("horizon_hours") != FORECAST_HORIZON_HOURS:
    raise ValueError("Feature metadata horizon does not match FORECAST_HORIZON_HOURS")
if metadata_target_columns != TARGET_COLUMNS:
    raise ValueError("Feature metadata target columns do not match the configured horizon")
FEATURE_COLUMNS = [
    column
    for column in feature_metadata["predictor_columns"]
    if (
        column.startswith(f"{station_id}__")
        and
        column.rsplit("__", maxsplit=1)[-1] not in EXCLUDED_METADATA_FIELDS
    )
]
if not FEATURE_COLUMNS:
    raise ValueError("The target station predictor contract is empty")

## Shared evaluation cohort

The model is fit and scored on rows from the joined feature artifacts. One row is one timestamp `t`, and it qualifies only when both conditions hold:

1. **Stage 3 marked the future window valid.** `{TARGET_STATION_ID}__target_valid` is true, and all 24 `{TARGET_STATION_ID}__target_t_plus_01` … `{TARGET_STATION_ID}__target_t_plus_24` values are present.
2. **Every model input is present.** The target station's raw water level at issue time `t`, engineered lag and rolling features, and raw weather/imputation inputs must all be available.

Train and test are filtered independently and are never pooled: the test artifact is sealed, and no statistic used by the model — not even a scaler mean — is ever computed from it.

## Helper functions

The eligibility helper validates the joined contract and returns rows with a valid 24-hour future window and complete predictors. The numeric conversion helper turns selected boolean/object predictors such as `imputed` columns into numeric values while leaving metadata columns out of the model matrix. The metric and preview helpers retain the direct 24-output order.

In [ ]:
def eligible_rows(frame: pd.DataFrame, *, station_id: str, artifact_name: str) -> pd.Series:
    """Return model-ready rows and reject incomplete joined artifacts."""
    required_columns = {"timestamp", TARGET_VALID_COLUMN, *FEATURE_COLUMNS, *TARGET_COLUMNS}
    missing = sorted(required_columns.difference(frame.columns))
    if missing:
        raise ValueError(
            f"{station_id} {artifact_name} artifact is missing required columns: {missing}"
        )

    target_valid_rows = frame[TARGET_VALID_COLUMN].eq(True)
    if frame.loc[target_valid_rows, TARGET_COLUMNS].isna().any(axis=None):
        raise ValueError(
            f"{station_id} {artifact_name} artifact has null targets in target-valid rows"
        )
    eligible = (
        target_valid_rows
        & frame[FEATURE_COLUMNS].notna().all(axis=1)
        & frame[TARGET_COLUMNS].notna().all(axis=1)
    )
    return eligible


def numeric_predictors(frame: pd.DataFrame) -> pd.DataFrame:
    """Convert the metadata-selected predictor columns to numeric values."""
    return frame[FEATURE_COLUMNS].apply(pd.to_numeric, errors="raise").astype(float)


def time_series_splits(n_rows: int) -> tuple[TimeSeriesSplit, list[tuple[np.ndarray, np.ndarray]], int]:
    """Build and validate expanding chronological CV folds."""
    initial_train_rows = int(n_rows * INITIAL_TRAIN_FRACTION)
    validation_budget = n_rows - initial_train_rows - EMBARGO_HOURS
    validation_test_size = validation_budget // N_VALIDATION_FOLDS
    if validation_test_size < 1:
        raise ValueError("Not enough eligible training rows for the configured CV policy")

    splitter = TimeSeriesSplit(
        n_splits=N_VALIDATION_FOLDS,
        gap=EMBARGO_HOURS,
        test_size=validation_test_size,
    )
    splits = list(splitter.split(np.arange(n_rows)))
    if len(splits) != N_VALIDATION_FOLDS:
        raise ValueError(f"Expected {N_VALIDATION_FOLDS} validation folds, got {len(splits)}")

    previous_validation_end = -1
    for fold_number, (fold_train_indices, fold_validation_indices) in enumerate(
        splits, start=1
    ):
        if fold_train_indices.size == 0 or fold_validation_indices.size == 0:
            raise ValueError(f"Fold {fold_number} is empty")
        if not np.array_equal(fold_train_indices, np.arange(fold_train_indices.size)):
            raise ValueError(f"Fold {fold_number} training rows are not chronological")
        if fold_validation_indices[0] - fold_train_indices[-1] - 1 != EMBARGO_HOURS:
            raise ValueError(f"Fold {fold_number} does not have the configured embargo")
        if fold_validation_indices[0] <= previous_validation_end:
            raise ValueError("Validation folds overlap or are out of order")
        previous_validation_end = fold_validation_indices[-1]
    return splitter, splits, validation_test_size


def select_alpha(cv_results: pd.DataFrame, metric: str) -> float:
    """Select the lowest mean CV metric, breaking ties by smaller alpha."""
    if metric not in {"mae", "rmse"}:
        raise ValueError("CV selection metric must be either 'mae' or 'rmse'")
    metric_column = f"{metric}_mean"
    if metric_column not in cv_results or cv_results.empty:
        raise ValueError(f"CV results do not contain {metric_column!r}")
    ranked = cv_results.sort_values([metric_column, "alpha"], kind="stable")
    return float(ranked.iloc[0]["alpha"])

In [ ]:
def prediction_preview(
    frame: pd.DataFrame, predictions: np.ndarray
) -> pd.DataFrame:
    """Return issue timestamps, 24 actual targets, and direct predictions."""
    predicted = pd.DataFrame(
        predictions,
        columns=[f"prediction_{target}" for target in TARGET_COLUMNS],
        index=frame.index,
    )
    return pd.concat([frame[["timestamp", *TARGET_COLUMNS]], predicted], axis=1)

## Load joined feature artifacts

Loads `all_stations_train_features.parquet` and `all_stations_test_features.parquet` from the joined Stage-3 directory. The metadata was loaded during setup and is checked before the fit, so a missing or incompatible artifact fails before any model work begins.

In [ ]:
train_path = PROCESSED_DIR / "all_stations_train_features.parquet"
test_path = PROCESSED_DIR / "all_stations_test_features.parquet"
for artifact_path in (METADATA_PATH, train_path, test_path):
    if not artifact_path.is_file():
        raise FileNotFoundError(f"Missing joined feature artifact: {artifact_path}")

train_features = pd.read_parquet(train_path)
test_features = pd.read_parquet(test_path)

## Apply the eligibility cohort

Builds the train and test masks with `eligible_rows()` and keeps only rows that pass. If either split has no eligible row, the notebook stops rather than fitting on an empty frame or reporting a metric computed from nothing.

In [ ]:
train_mask = eligible_rows(
    train_features, station_id=station_id, artifact_name="train"
)
test_mask = eligible_rows(
    test_features, station_id=station_id, artifact_name="test"
)
if not train_mask.any():
    raise ValueError(f"{station_id} train artifact has no eligible model rows")
if not test_mask.any():
    raise ValueError(f"{station_id} test artifact has no eligible scoring rows")

train_rows = (
    train_features.loc[train_mask]
    .sort_values("timestamp", kind="mergesort")
    .reset_index(drop=True)
)
test_rows = (
    test_features.loc[test_mask]
    .sort_values("timestamp", kind="mergesort")
    .reset_index(drop=True)
)
if not train_rows["timestamp"].is_monotonic_increasing:
    raise ValueError("Eligible training rows are not chronological")

## Time-series alpha search

Eligible training rows are sorted by issue time before `TimeSeriesSplit` creates five expanding-window folds. The explicit `test_size` allocates the post-initial-training portion across the folds, while the 24-row gap acts as the requested hourly embargo. Each alpha has one parent MLflow run and each fold has one nested child run.

Every fold fits its own `StandardScaler` and 24-output `Ridge` model using only that fold's training rows. The sealed test cohort is not referenced until the final fit below.

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
cv_splitter, cv_splits, validation_test_size = time_series_splits(len(train_rows))
cv_results_rows = []

for alpha in RIDGE_ALPHAS:
    fold_aggregate_rows = []
    fold_horizon_rows = []
    with mlflow.start_run(
        run_name=f"ridge_cv_alpha_{alpha:g}",
        nested=False,
        tags={"phase": "cv", "run_type": "alpha_parent"},
    ):
        mlflow.log_params({
            "phase": "cv",
            "run_type": "alpha_parent",
            "alpha": alpha,
            "n_validation_folds": N_VALIDATION_FOLDS,
            "validation_test_size": validation_test_size,
            "embargo_hours": EMBARGO_HOURS,
            "selection_metric": CV_SELECTION_METRIC,
            "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
        })

        for fold_number, (fold_train_indices, fold_validation_indices) in enumerate(
            cv_splits, start=1
        ):
            fold_train_rows = train_rows.iloc[fold_train_indices]
            fold_validation_rows = train_rows.iloc[fold_validation_indices]
            fold_scaler = StandardScaler()
            fold_train_predictors = fold_scaler.fit_transform(
                numeric_predictors(fold_train_rows)
            )
            fold_validation_predictors = fold_scaler.transform(
                numeric_predictors(fold_validation_rows)
            )
            fold_ridge = Ridge(alpha=alpha)
            fold_ridge.fit(fold_train_predictors, fold_train_rows[TARGET_COLUMNS])
            fold_predictions = np.asarray(
                fold_ridge.predict(fold_validation_predictors)
            )
            if fold_predictions.shape != (len(fold_validation_rows), len(TARGET_COLUMNS)):
                raise ValueError(f"Unexpected fold prediction shape: {fold_predictions.shape}")
            if not np.isfinite(fold_predictions).all():
                raise ValueError("Ridge produced non-finite fold predictions")

            fold_aggregate, fold_per_horizon = metric_tables(
                fold_validation_rows[TARGET_COLUMNS],
                fold_predictions,
                target_columns=TARGET_COLUMNS,
                station_id=station_id,
            )
            fold_aggregate_rows.append(fold_aggregate.iloc[0])
            fold_horizon_rows.append(fold_per_horizon)
            with mlflow.start_run(
                run_name=f"ridge_cv_alpha_{alpha:g}_fold_{fold_number}",
                nested=True,
                tags={"phase": "cv", "run_type": "fold", "fold": str(fold_number)},
            ):
                mlflow.log_params({
                    "phase": "cv",
                    "run_type": "fold",
                    "alpha": alpha,
                    "fold": fold_number,
                    "train_rows": len(fold_train_rows),
                    "validation_rows": len(fold_validation_rows),
                    "gap_rows": EMBARGO_HOURS,
                    "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
                    "train_end": fold_train_rows["timestamp"].iloc[-1].isoformat(),
                    "validation_start": fold_validation_rows["timestamp"].iloc[0].isoformat(),
                    "validation_end": fold_validation_rows["timestamp"].iloc[-1].isoformat(),
                })
                mlflow.log_metrics({
                    "mae": float(fold_aggregate.iloc[0]["mae"]),
                    "rmse": float(fold_aggregate.iloc[0]["rmse"]),
                    **{
                        f"mae_horizon_{row.horizon_hours:02d}": float(row.mae)
                        for row in fold_per_horizon.itertuples()
                    },
                    **{
                        f"rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
                        for row in fold_per_horizon.itertuples()
                    },
                })

        fold_aggregate_metrics = pd.DataFrame(fold_aggregate_rows)
        fold_horizon_metrics = pd.concat(fold_horizon_rows, ignore_index=True)
        parent_metrics = {
            "cv_mae_mean": float(fold_aggregate_metrics["mae"].mean()),
            "cv_mae_std": float(fold_aggregate_metrics["mae"].std(ddof=0)),
            "cv_rmse_mean": float(fold_aggregate_metrics["rmse"].mean()),
            "cv_rmse_std": float(fold_aggregate_metrics["rmse"].std(ddof=0)),
        }
        for horizon, horizon_metrics in fold_horizon_metrics.groupby("horizon_hours"):
            parent_metrics[f"cv_mae_horizon_{horizon:02d}_mean"] = float(horizon_metrics["mae"].mean())
            parent_metrics[f"cv_mae_horizon_{horizon:02d}_std"] = float(horizon_metrics["mae"].std(ddof=0))
            parent_metrics[f"cv_rmse_horizon_{horizon:02d}_mean"] = float(horizon_metrics["rmse"].mean())
            parent_metrics[f"cv_rmse_horizon_{horizon:02d}_std"] = float(horizon_metrics["rmse"].std(ddof=0))
        mlflow.log_metrics(parent_metrics)
        cv_results_rows.append({
            "alpha": alpha,
            "mae_mean": parent_metrics["cv_mae_mean"],
            "mae_std": parent_metrics["cv_mae_std"],
            "rmse_mean": parent_metrics["cv_rmse_mean"],
            "rmse_std": parent_metrics["cv_rmse_std"],
        })

cv_results = pd.DataFrame(cv_results_rows)
selected_alpha = select_alpha(cv_results, CV_SELECTION_METRIC)
print(f"Selected Ridge alpha by CV {CV_SELECTION_METRIC.upper()}: {selected_alpha:g}")
display(cv_results)

## Retrain the selected alpha

The selected alpha is retrained once on all eligible, chronologically ordered training rows. The scaler is fitted on those training predictors only and then applied to the sealed test predictors.

In [ ]:
final_scaler = StandardScaler()
final_train_predictors = final_scaler.fit_transform(numeric_predictors(train_rows))
final_test_predictors = final_scaler.transform(numeric_predictors(test_rows))
final_ridge = Ridge(alpha=selected_alpha)
final_ridge.fit(final_train_predictors, train_rows[TARGET_COLUMNS])
test_predictions = np.asarray(
    final_ridge.predict(final_test_predictors)
)
if test_predictions.shape != (len(test_rows), len(TARGET_COLUMNS)):
    raise ValueError(f"Unexpected prediction shape: {test_predictions.shape}")
if not np.isfinite(test_predictions).all():
    raise ValueError("Ridge produced non-finite predictions")

## Evaluate on the test cohort

A single scoring pass over the sealed test cohort reports aggregate MAE/RMSE, the same metrics for each lead in the direct 24-hour forecast, and a short preview for comparison with actual targets. There is no second pass and no refitting.

In [ ]:
aggregate_metrics, per_horizon_metrics = metric_tables(
    test_rows[TARGET_COLUMNS],
    test_predictions,
    target_columns=TARGET_COLUMNS,
    station_id=station_id,
)
if not np.isfinite(aggregate_metrics[["mae", "rmse"]].to_numpy()).all():
    raise ValueError("Ridge reported non-finite aggregate metrics")
if not np.isfinite(per_horizon_metrics[["mae", "rmse"]].to_numpy()).all():
    raise ValueError("Ridge reported non-finite horizon metrics")
with mlflow.start_run(
    run_name=f"ridge_test_alpha_{selected_alpha:g}",
    nested=False,
    tags={"phase": "test", "run_type": "sealed_test"},
):
    mlflow.log_params({
        "phase": "test",
        "run_type": "sealed_test",
        "alpha": selected_alpha,
        "selection_metric": CV_SELECTION_METRIC,
        "forecast_horizon_hours": FORECAST_HORIZON_HOURS,
        "cv_selected_metric": float(cv_results.loc[cv_results["alpha"].eq(selected_alpha), f"{CV_SELECTION_METRIC}_mean"].iloc[0]),
        "scored_issue_times": len(test_rows),
    })
    mlflow.log_metrics({
        "mae": float(aggregate_metrics.iloc[0]["mae"]),
        "rmse": float(aggregate_metrics.iloc[0]["rmse"]),
        **{
            f"mae_horizon_{row.horizon_hours:02d}": float(row.mae)
            for row in per_horizon_metrics.itertuples()
        },
        **{
            f"rmse_horizon_{row.horizon_hours:02d}": float(row.rmse)
            for row in per_horizon_metrics.itertuples()
        },
    })
print(f"Ridge test results for {station_id} (selected alpha={selected_alpha:g})")
display(aggregate_metrics)
display(per_horizon_metrics)
display(prediction_preview(test_rows, test_predictions).head(PREDICTION_PREVIEW_ROWS))